# Exploração e coleta dos dados de votações da Câmara

Este notebook documenta a exploração inicial dos dados abertos da Câmara dos Deputados
usados no projeto O Gabinete. Aqui baixamos os arquivos brutos, inspecionamos sua estrutura
e geramos as versões limpas usadas no cálculo de similaridade entre deputados.

A lógica final de produção está nos arquivos `coleta.py` e `processamento.py`, dentro de
`pipeline/`. Este notebook serve como registro exploratório e visual do processo.


In [1]:
import requests
import pandas as pd
from pathlib import Path

## Definindo o período e as fontes de dados

Como o grupo ainda não fechou oficialmente o período de análise, usamos 2024 como
referência provisória (item #2 do board).

Três fontes são usadas:
- `votacoesVotos`: o voto individual de cada deputado em cada votação
- `votacoes`: metadados de cada votação (órgão e descrição), usados para filtrar as votações de mérito
- `votacoesProposicoes`: as proposições ligadas a cada votação, usadas para filtrar pelo tipo (PL, PEC...)

In [2]:
ANO_REFERENCIA = 2024

url_votos = f'https://dadosabertos.camara.leg.br/arquivos/votacoesVotos/csv/votacoesVotos-{ANO_REFERENCIA}.csv'
url_votacoes = f'https://dadosabertos.camara.leg.br/arquivos/votacoes/csv/votacoes-{ANO_REFERENCIA}.csv'
url_proposicoes = f'https://dadosabertos.camara.leg.br/arquivos/votacoesProposicoes/csv/votacoesProposicoes-{ANO_REFERENCIA}.csv'

## Baixando os dados brutos

Verifica se os arquivos já existem em `dados/brutos/` antes de baixar, para não repetir
o download toda vez que o notebook rodar. Os caminhos são calculados a partir da raiz do
projeto (via `encontrar_raiz`), então funciona independente de onde o notebook estiver salvo.

In [3]:
def encontrar_raiz(marcador="requirements.txt"):
    caminho = Path.cwd()
    while not (caminho / marcador).exists():
        caminho = caminho.parent
    return caminho

RAIZ = encontrar_raiz()
pasta = RAIZ / "dados" / "brutos"
pasta.mkdir(parents=True, exist_ok=True)

for url in [url_votos, url_votacoes, url_proposicoes]:
    nome_arquivo = url.split('/')[-1]
    caminho_arquivo = pasta / nome_arquivo

    if not caminho_arquivo.exists():
        print(f'Baixando {nome_arquivo}...')
        response = requests.get(url)
        with open(caminho_arquivo, 'wb') as f:
            f.write(response.content)
    else:
        print(f'{nome_arquivo} já existe. Pulando download.')

with open(pasta / f'votacoesVotos-{ANO_REFERENCIA}.csv', 'r', encoding='utf-8') as f:
    print(f.readline())

votacoesVotos-2024.csv já existe. Pulando download.
votacoes-2024.csv já existe. Pulando download.
votacoesProposicoes-2024.csv já existe. Pulando download.
﻿"idVotacao";"uriVotacao";"dataHoraVoto";"voto";"deputado_id";"deputado_uri";"deputado_nome";"deputado_siglaPartido";"deputado_uriPartido";"deputado_siglaUf";"deputado_idLegislatura";"deputado_urlFoto"



### Observação sobre o formato do CSV

O cabeçalho impresso acima revela dois detalhes importantes do arquivo da Câmara:
- separador é `;`, não vírgula (por isso o `sep=';'` na leitura)
- o arquivo tem um BOM no início (por isso o `encoding='utf-8-sig'`, que remove esse
  caractere invisível automaticamente)

## Lendo o CSV de votos para o pandas

In [4]:
df = pd.read_csv(pasta / f'votacoesVotos-{ANO_REFERENCIA}.csv', sep=';', encoding='utf-8-sig')

## Limpando o arquivo de votos

O arquivo bruto traz dados do deputado (nome, partido, UF, foto) repetidos em toda linha,
o que é redundante e deixa o arquivo maior do que precisa. Mantemos só o essencial para o
cálculo de similaridade: `idVotacao`, `deputado_id` e `voto`, além de `siglaPartido` e
`siglaUf` (usados depois para colorir os vértices do grafo).

O resultado é salvo em `dados/processado/votos-{ano}-limpo.csv`.

In [5]:
colunas_uteis = ['idVotacao', 'deputado_id', 'voto', 'deputado_siglaPartido', 'deputado_siglaUf']
df_limpo = df[colunas_uteis]

(RAIZ / 'dados' / 'processado').mkdir(parents=True, exist_ok=True)
df_limpo.to_csv(RAIZ / 'dados' / 'processado' / f'votos-{ANO_REFERENCIA}-limpo.csv', index=False)


## Gerando a tabela de deputados

Como os dados do deputado se repetem em cada linha de voto, extraímos um registro único
por `deputado_id` a partir do próprio arquivo bruto, incluindo a URL da foto (que já vem
válida no CSV, sem precisar de chamada extra à API).

Resultado salvo em `dados/processado/deputados.csv`.

In [6]:
deputados_df = df.drop_duplicates(subset='deputado_id')[
    ['deputado_id', 'deputado_nome', 'deputado_siglaPartido', 'deputado_siglaUf', 'deputado_urlFoto']
].copy()

deputados_df.to_csv(RAIZ / 'dados' / 'processado' / 'deputados.csv', index=False)

## Filtrando as votações de mérito

O arquivo de votos traz toda votação nominal do ano, mas para a similaridade só interessam as
votações sobre o **conteúdo** das proposições. O filtro tem três etapas:

1. **Tipo de proposição**: só tipos legislativos (PL, PEC, MPV, PLP, PDC, PDL, PLN), descartando
   requerimentos (REQ, RIC), concessões de rádio/TV (TVR) etc.
2. **Plenário**: votações de comissão têm só 20–60 votantes e afastariam deputados que apenas não
   estão na mesma comissão.
3. **Sem votações procedimentais**: `votacoesProposicoes` liga a votação à proposição *afetada*,
   então um "Requerimento de Retirada de Pauta do PL X" também aparece como PL. Esses casos são
   removidos pela `descricao` da votação.

Primeiro, a distribuição dos tipos de proposição votados no ano:

In [7]:
df_proposicoes = pd.read_csv(pasta / f'votacoesProposicoes-{ANO_REFERENCIA}.csv', sep=';', encoding='utf-8-sig')

df_proposicoes['proposicao_siglaTipo'].value_counts()

proposicao_siglaTipo
PL     4367
REQ     754
PDL     710
TVR     487
RIC     460
PLP     306
MPV     109
PEC      83
MSC      62
PRC      40
SUG      20
PLN      16
PDC       6
REC       6
INC       5
CMC       3
SLD       3
REP       2
REL       2
RPD       1
SOR       1
Name: count, dtype: int64

In [8]:
TIPOS_DE_MERITO = ['PL', 'PEC', 'MPV', 'PLP', 'PDC', 'PDL', 'PLN']
PADRAO_PROCEDIMENTAL = r'requerimento|retirada de pauta|adiamento|encerramento'

df_votacoes = pd.read_csv(pasta / f'votacoes-{ANO_REFERENCIA}.csv', sep=';', encoding='utf-8-sig')

ids_tipo = df_proposicoes.loc[df_proposicoes['proposicao_siglaTipo'].isin(TIPOS_DE_MERITO), 'idVotacao'].unique()
com_voto_nominal = df_votacoes['id'].isin(ids_tipo) & df_votacoes['id'].isin(df_limpo['idVotacao'])
plenario = df_votacoes['siglaOrgao'] == 'PLEN'
procedimental = df_votacoes['descricao'].fillna('').str.contains(PADRAO_PROCEDIMENTAL, case=False)

print('Tipos de mérito com voto nominal:', com_voto_nominal.sum())
print('  ...só no Plenário:             ', (com_voto_nominal & plenario).sum())
print('  ...sem procedimentais:         ', (com_voto_nominal & plenario & ~procedimental).sum())

votacoes_de_merito = df_votacoes.loc[com_voto_nominal & plenario & ~procedimental, 'id'].unique()

Tipos de mérito com voto nominal: 425
  ...só no Plenário:              270
  ...sem procedimentais:          141


Amostra das descrições que ficaram, para conferir que são mesmo votações de mérito
(texto principal, substitutivos, emendas e destaques):

In [9]:
pd.set_option('display.max_colwidth', 150)
df_votacoes.loc[df_votacoes['id'].isin(votacoes_de_merito), 'descricao'].sample(10, random_state=1)

5393                                                                                             Mantido o texto. Sim: 283; não: 64; abstenção: 2; total: 349.
10280    Aprovado o Substitutivo Reformulado ao Projeto de Lei nº 1.637, de 2019, adotado pelo relator da Comissão de Constituição e Justiça e de Cidadania...
1368     Aprovado o Substitutivo ao Projeto de Lei nº 1.548, de 2022, adotado pelo relator da Comissão de Finanças e Tributação. Sim: 383; não: 19; total: ...
231      Rejeitado o Recurso nº 3/2024, contra Parecer Terminativo da Comissão de Finanças e Tributação às Emendas de Plenário, de 2024. Sim: 139; não: 290...
10315                                                                         Aprovada a Emenda de Redação n° 2. Sim: 314; Não: 117; Abstenção: 1; Total: 432.
2989                                                                                                          Mantido o texto. Sim: 315; não: 120; total: 435.
307                                           

Resultado salvo em `dados/processado/votos-{ano}-merito.csv`, que é a entrada do cálculo de
similaridade por cosseno (`pipeline/similaridade.py`).

In [10]:
df_merito = df_limpo[df_limpo['idVotacao'].isin(votacoes_de_merito)]
df_merito.to_csv(RAIZ / 'dados' / 'processado' / f'votos-{ANO_REFERENCIA}-merito.csv', index=False)

print(df_merito.shape)
df_merito['voto'].value_counts()

(57540, 5)


voto
Sim          38064
Não          18868
Obstrução      355
Artigo 17      129
Abstenção      124
Name: count, dtype: int64